In [ ]:
import time
from googleapiclient.discovery import build
api_key = ''
youtube = build('youtube', 'v3', developerKey=api_key)
video_id="vXE6D-bthVE"

**Subtitle Download with the Official API**

**Method 1 for Extracting Video Information**

In [ ]:
youtube = build("youtube", "v3", developerKey=api_key)

request = youtube.videos().list(
    part="snippet",
    id=video_id
)
response = request.execute()

if response and "items" in response and len(response["items"]) > 0:
    snippet = response["items"][0]["snippet"]
    title = snippet.get("title")
    description = snippet.get("description")
    tags = snippet.get("tags") if "tags" in snippet else []

    print(f"Título: {title}")
    print(f"Descripción:\n{description}")
    print(f"Etiquetas: {tags}")


**Creating the Comment Retrieval Pipeline**

In [ ]:
# QUERY TEMPLATE - FROM: https://medium.com/mcd-unison/youtube-data-api-v3-in-python-tutorial-with-examples-e829a25d2ebd
import googleapiclient.discovery
from youtube_transcript_api import YouTubeTranscriptApi
import time
from googleapiclient.discovery import build

api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY  = ''
youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey = DEVELOPER_KEY)
request = youtube.search().list()
response = request.execute()
print(response)

In [ ]:
import googleapiclient.discovery
from youtube_transcript_api import YouTubeTranscriptApi
import time
from googleapiclient.discovery import build
def display_yt_response(search_response):
    for search_result in search_response.search_results:
        print(f'Video ID: {search_result.video_id}\n')

In [ ]:
api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY  = ''
youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey = DEVELOPER_KEY)
request = youtube.search().list(
    part="id",
    maxResults=25,
    q="Reiki",
    videoCaption="closedCaption",
    type="video"
)
response = request.execute()

In [ ]:
video_ids = [item["id"]["videoId"] for item in response["items"] if "videoId" in item["id"]]
video_ids

In [ ]:
basic_info_dict = {}
for video_id in video_ids:
    try:
        request = youtube.videos().list(
            part="snippet",
            id=video_id
        )
        response = request.execute()

        if response and "items" in response and len(response["items"]) > 0:
            snippet = response["items"][0]["snippet"]
            title = snippet.get("title")
            description = snippet.get("description")
            tags = snippet.get("tags") if "tags" in snippet else []

            video_info = {
                "title": title,
                "description": description,
                "tags": tags
            }
            basic_info_dict[video_id] = video_info
    except:
        pass

In [ ]:
import requests
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
import os
GOOGLE_API_KEY=os.getenv('')

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
model = genai.GenerativeModel('gemini-1.5-flash')

In [ ]:
genai.configure(api_key='')

In [ ]:
prompt_base = """
A continuación recibirás la información básica de un vídeo de youtube que deberás usar para clasificar el vídeo en una de las siguientes categorías:

Científico: El vídeo presenta información basada en evidencia empírica, métodos científicos y consenso científico.
Pseudocientífico: El vídeo presenta información que se disfraza de ciencia, pero carece de evidencia empírica, métodos científicos rigurosos o consenso científico.
Irrelevante: El contenido del vídeo no está relacionado con ciencia o pseudociencia, o es difícil de clasificar debido a su ambigüedad o falta de claridad. Si la transcripción es extremadamente corta y consiste en una mera descripción genérica, clasifícalo como irrelevante.

Devuelve únicamente la etiqueta de la categoría. Escribe exclusivamente la palabra, no quiero que haya nada más tras ella. Ni siquiera un salto de linea o un punto.

A continuación tienes el título del vídeo, la descripción del mismo, y sus etiquetas. En ocasiones alguno de estos campos puede estar vacío. En este caso haz la evaluación con la información que tienes disponible. Si no hay información suficiente para determinar con precision la categoría, simplemente clasificalo como "Irrelevante".

Título: {}

Descripción del vídeo: {}

Etiquetas del vídeo: {}
"""

In [ ]:
for video_id, info in basic_info_dict.items():
    print(info['title'])

In [ ]:
dict_prompt_info = {}

for video_id, transcripcion in basic_info_dict.items():
    prompt_con_transcripcion = prompt_base.format(transcripcion['title'], transcripcion['description'], transcripcion['tags'])
    dict_prompt_info[video_id] = prompt_con_transcripcion


In [ ]:
import time

dict_respuestas_llm = {}
consulta_contador = 0
for video_id, prompt_con_transcripcion in dict_prompt_info.items():
    response = model.generate_content(prompt_con_transcripcion)
    dict_respuestas_llm[video_id] = response.text
    consulta_contador += 1
    if consulta_contador % 15 == 0:
        print("Pausando por un minuto...")
        time.sleep(60)
        print("Continuando...")

In [ ]:
dict_respuestas_llm

In [ ]:
dict_respuestas_llm_filtrado = {clave: valor for clave, valor in dict_respuestas_llm.items() if "Irrelevante\n" not in valor}
dict_respuestas_llm_filtrado

**Obtaining video comments**

In [ ]:
import time
from googleapiclient.discovery import build
youtube = build('youtube', 'v3', developerKey=api_key)

# Function to retrieve comments from a YouTube video
def retrieve_comments(video_id):
    comments_data = []  # List to store comments and related data
    next_page_token = None  # Token for paginated API responses

    while True:
        try:
            # Construct a request to retrieve comment threads for the video
            comments_request = youtube.commentThreads().list(
                part='snippet',
                videoId=video_id,
                textFormat='plainText',
                maxResults=100,  # Maximum number of comments per page
                pageToken=next_page_token  # Use token for pagination
            )
            # Execute the comments request and store the response
            comments_response = comments_request.execute()
    
            # Iterate through comments in the response
            for item in comments_response['items']:
                comment_snippet = item['snippet']['topLevelComment']['snippet']
                comment_text = comment_snippet['textDisplay']  # Text of the comment
                comment_likes = comment_snippet['likeCount']  # Number of likes on the comment
                comment_timestamp = comment_snippet['publishedAt']  # Timestamp of the comment
    
                replies_data = []  # List to store reply texts and their timestamps
                # Construct a request to retrieve replies to the current comment
                reply_request = youtube.comments().list(
                    part='snippet',
                    parentId=item['id'],  # ID of the current comment
                    maxResults=100  # Maximum number of replies per page
                )
                # Execute the reply request and store the response
                reply_response = reply_request.execute()
    
                # Iterate through replies in the response
                for reply_item in reply_response['items']:
                    reply_text = reply_item['snippet']['textDisplay']
                    reply_timestamp = reply_item['snippet']['publishedAt']  # Timestamp of the reply
                    replies_data.append({
                        'reply': reply_text,
                        'timestamp': reply_timestamp
                    })
    
                # Store comment data and related replies in the comments_data list
                comments_data.append({
                    'comment': comment_text,
                    'likes': comment_likes,
                    'timestamp': comment_timestamp,
                    'replies': replies_data
                })
    
            # Retrieve the next page token from the response
            next_page_token = comments_response.get('nextPageToken')
            # Break the loop if there are no more pages or a maximum of 500 comments are retrieved
            if not next_page_token or len(comments_data) >= 500:
                break
    
            time.sleep(2)  # Add a delay between API requests to avoid rate limits
        except:
            return []

    return comments_data

In [ ]:
import pandas as pd
# Dictionary where each key is a video and each value is a dataframe with the comments and timestamps
video_comments_dict = {}

# Iterate over the items in the original dictionary
for clave, valor in dict_respuestas_llm_filtrado.items():
    # Retrieve comments
    comments = retrieve_comments(clave)
    
    # Create a new object containing only each comment and its timestamp
    comments_with_timestamps = []
    for comment in comments:
        comments_with_timestamps.append({
            'comment': comment['comment'],
            'timestamp': comment['timestamp']
        })
    
    # Convert the object into a pandas dataframe
    dataframe = pd.DataFrame(comments_with_timestamps)
    
    # Add the dataframe to the new dictionary using the same key
    video_comments_dict[clave] = dataframe